In [ ]:
# import required packages

import pandas as pd
import os


In [ ]:
# functions for importing and processing the data

def process_cpi_data(file_names: dict[str, str]) -> pd.DataFrame:
    """
    Import and clean the raw cpi data from the csv files and return them in a combined dataframe.
    """

    # prepare the dictionaries to store the dataframes and convert the months to numerical values
    data = []
    months_mapping = {
                'januari': '01', 'februari': '02', 'maart': '03', 'april': '04',
                'mei': '05', 'juni': '06', 'juli': '07', 'augustus': '08',
                'september': '09', 'oktober': '10', 'november': '11', 'december': '12'
            }

    # loop through the file names and import the data
    for file_key, file_name in file_names.items():
        file_path = os.path.join("..", "data", "raw_data", file_name)
        df = pd.read_csv(file_path, delimiter=';', encoding='utf-8', skiprows=4)
        df = df.iloc[:-1, :]    # delete last row

        # convert dates to datetime
        date_clean = df.iloc[:, 0].replace(months_mapping, regex=True)
        df['Date'] = pd.to_datetime(date_clean, format='%Y %m')

        # convert data to floats
        df[f'CPI_{file_key}'] = (
            df.iloc[:, 1]
            .astype(str)
            .str.replace(',', '.', regex=False)
            .str.strip()
            .astype(float)
        )

        data.append(df[['Date', f'CPI_{file_key}']].set_index('Date'))

    combined_df = pd.concat(data, axis=1).reset_index()
    return combined_df

def process_mode_data(file_name: str) -> pd.DataFrame:
    """
    Import and clean the raw mode data from the csv file and return it as a dataframe.
    Translates column and index names to English and converts the dates to datetime format.
    Calculates daily distance and daily travel time per person for each mode.
    """

    # import the csv as a dataframe
    file = os.path.join("..", "data", "raw_data", file_name)
    df = pd.read_csv(file, delimiter=';', encoding='utf-8', skiprows=5, header=[0, 1, 2], index_col=[0, 1])
    df = df.iloc[:-1, :]                            # delete last row
    df.columns.names = [None] * df.columns.nlevels
    df.index.names = ['Date', 'Travel purpose']     # rename index

    # convert dates to datetime and fix index
    years = pd.to_datetime(
        df.index.get_level_values('Date').str.replace('*', '', regex=False).astype(int),
        format='%Y'
    )
    df.index = pd.MultiIndex.from_arrays(
        [years, df.index.get_level_values('Travel purpose')],
        names=['Date', 'Travel purpose']
    )

    # convert data to floats
    df = df.map(lambda x: str(x).replace(',', '.').strip() if pd.notna(x) else x)
    df = df.apply(pd.to_numeric, errors='coerce')

    # translate travel purposes in index
    purpose_mapping = {
        'Totaal': 'Total',
        'Van en naar het werk': 'Commuting',
        'Winkelen, boodschappen doen': 'Shopping',
        'Onderwijs volgen, cursus, kinderopvang': 'Education',
        'Vrije tijd': 'Leisure',
        'Overige reismotieven': 'Other'
    }
    cleaned_purposes = df.index.get_level_values('Travel purpose').map(
        lambda p: purpose_mapping.get(p, p)
    )
    df.index = pd.MultiIndex.from_arrays(
        [df.index.get_level_values('Date'), cleaned_purposes],
        names=['Date', 'Travel purpose']
    )

    # dictionaries for translating modes, indicators and units to English
    mode_mapping = {
        'Totaal': 'Total',
        'Auto (bestuurder)': 'Car (driver)',
        'Auto (passagier)': 'Car (passenger)',
        'Trein': 'Train',
        'Bus/tram/metro': 'Bus/tram/metro',
        'Fiets': 'Bicycle',
        'Lopen': 'Walking',
        'Overige vervoerwijze': 'Other modes'
    }

    indicator_mapping = {
        'Ritten per persoon per dag (gemiddeld)': 'Trips per person per day',
        'Afgelegde afstand per rit (gemiddeld)': 'Distance per trip',
        'Reisduur per rit': 'Travel time per trip'
    }

    unit_mapping = {
        'aantal': 'trips',
        'km': 'km',
        'Minuten': 'min'
    }

    # flatten from 3 header layers to 2 layers (incorporating unit in parentheses)
    new_columns = []
    for mode, indicator, unit in df.columns:
        trans_mode = mode_mapping.get(mode, mode)
        trans_indicator = indicator_mapping.get(indicator, indicator)
        trans_unit = unit_mapping.get(unit, unit)
        new_columns.append((trans_mode, f"{trans_indicator} ({trans_unit})"))

    df.columns = pd.MultiIndex.from_tuples(new_columns, names=['Mode', 'Indicator'])

    # add 'Car (total)' mode
    driver = 'Car (driver)'
    passenger = 'Car (passenger)'
    total_car = 'Car (total)'

    if driver in df.columns.get_level_values('Mode') and passenger in df.columns.get_level_values('Mode'):
        # Daily trips for driver and passenger
        trips_d = df[(driver, 'Trips per person per day (trips)')]
        trips_p = df[(passenger, 'Trips per person per day (trips)')]
        trips_tot = trips_d + trips_p
        df[(total_car, 'Trips per person per day (trips)')] = trips_tot

        # Weighted averages for distance and time per trip
        dist_d = df[(driver, 'Distance per trip (km)')]
        dist_p = df[(passenger, 'Distance per trip (km)')]
        df[(total_car, 'Distance per trip (km)')] = ((trips_d * dist_d + trips_p * dist_p) / trips_tot).fillna(0)

        time_d = df[(driver, 'Travel time per trip (min)')]
        time_p = df[(passenger, 'Travel time per trip (min)')]
        df[(total_car, 'Travel time per trip (min)')] = ((trips_d * time_d + trips_p * time_p) / trips_tot).fillna(0)

    # calculate daily distance and daily travel time per person for each mode
    all_modes = list(dict.fromkeys(df.columns.get_level_values('Mode')))
    for m in all_modes:
        trips_col = (m, 'Trips per person per day (trips)')
        dist_col = (m, 'Distance per trip (km)')
        time_col = (m, 'Travel time per trip (min)')

        if trips_col in df.columns and dist_col in df.columns:
            df[(m, 'Distance per person per day (km)')] = df[trips_col] * df[dist_col]

        if trips_col in df.columns and time_col in df.columns:
            df[(m, 'Travel time per person per day (min)')] = df[trips_col] * df[time_col]

    # sort columns alphabetically by mode
    df = df.sort_index(axis=1)

    return df

def export_dataframe(df: pd.DataFrame, file_name: str, index: bool = False) -> None:
    """
    Export the dataframe to a csv file in the processed_data folder.
    """
    file_path = os.path.join("..", "data", "processed_data", file_name)
    df.to_csv(file_path, index=index, sep=';', encoding='utf-8')


In [ ]:
# specify the file names and execute the functions

# specify the file names
cpi_data_file_names = {
    "Train": "83131NED_TabelIndeling_20261002110845 - CPI Train.csv",
    "Diesel": "83131NED_TabelIndeling_20261002111330 - CPI Diesel.csv",
    "Petrol": "83131NED_TabelIndeling_20261002111353 - CPI Petrol.csv",
    "LPG": "83131NED_TabelIndeling_20261002111410 - CPI LPG.csv"
}
mode_data_file_name = "84755NED_TabelIndeling_20261007110933 - mode data.csv"

# process and export the data
export_dataframe(process_cpi_data(cpi_data_file_names), "cpi_data.csv")
export_dataframe(process_mode_data(mode_data_file_name), "mode_data.csv", index=True)


In [ ]:
# example of how to import the processed data

# specify file paths
cpi_file = os.path.join("..", "data", "processed_data", 'cpi_data.csv')
cpi_df = pd.read_csv(cpi_file, delimiter=';', encoding='utf-8', index_col=[0])

# import data
mode_file = os.path.join("..", "data", "processed_data", 'mode_data.csv')
mode_df = pd.read_csv(mode_file, delimiter=';', encoding='utf-8', header=[0, 1], index_col=[0, 1])

# display dataframes
display(cpi_df)
display(mode_df)